In [7]:
import requests
import json
import numpy as np

# ── Configuration ──
FDA_ONLY = True  # Set to False to include all drugs (not just FDA-approved)

# Build query based on FDA filter
if FDA_ONLY:
    query = {
        "query": {
            "type": "terminal",
            "service": "text_chem",
            "parameters": {
                "attribute": "drugbank_info.drug_groups",
                "operator": "exact_match",
                "value": "approved",
                "negation": False
            }
        },
        "return_type": "mol_definition",
        "request_options": {
            "paginate": {"start": 0, "rows": 3000},
            "sort": [{"sort_by": "chem_comp.formula_weight", "direction": "desc"}]
        }
    }
else:
    query = {
        "query": {
            "type": "terminal",
            "service": "text_chem",
            "parameters": {
                "attribute": "drugbank_container_identifiers.drugbank_id",
                "operator": "exists",
                "negation": False
            }
        },
        "return_type": "mol_definition",
        "request_options": {
            "paginate": {"start": 0, "rows": 3000},
            "sort": [{"sort_by": "chem_comp.formula_weight", "direction": "desc"}]
        }
    }

url = "https://search.rcsb.org/rcsbsearch/v2/query"
response = requests.post(url, json=query)
results = response.json()

comp_ids = [r["identifier"] for r in results.get("result_set", [])]
label = "FDA-approved" if FDA_ONLY else "all DrugBank"
print(f"Found {len(comp_ids)} {label} drugs")
print("First 10:", comp_ids[:10])

Found 988 FDA-approved drugs
First 10: ['B1Z', 'PRD_900028', 'PRD_000204', 'BLM', 'CNC', 'COB', 'IDB', 'QWP', 'FI8', 'A4I']


In [8]:
import requests
import os
import time
import numpy as np
from rdkit import Chem, RDLogger
from rdkit.Chem import Descriptors, rdMolDescriptors
from scipy.spatial.distance import pdist

# Suppress noisy RDKit warnings (2D/3D tagging, valence issues)
RDLogger.DisableLog('rdApp.*')

os.makedirs("ligands_sdf_large", exist_ok=True)

# ── Configuration ──
FDA_ONLY = True             # Set to False to include all DrugBank drugs
REQUIRE_COMPLEX = False      # True = large AND complex; False = large only
NUM_LIGANDS = 335           # Number of largest drugs to keep
MIN_MW = 100                # Minimum formula weight (Da)
MIN_HEAVY_ATOMS = 15        # Minimum heavy atoms for "large"
MIN_EXTENT = 15.0           # Minimum spatial extent (Å)
MIN_ROTATABLE_BONDS = 3     # Minimum rotatable bonds for "complex" (only if REQUIRE_COMPLEX)
MIN_RINGS = 2              # Minimum ring count for "complex" (only if REQUIRE_COMPLEX)

# Step 1: Query RCSB for drugs with high MW
fda_node = {
    "type": "terminal",
    "service": "text_chem",
    "parameters": {
        "attribute": "drugbank_info.drug_groups",
        "operator": "exact_match",
        "value": "approved",
        "negation": False
    }
} if FDA_ONLY else {
    "type": "terminal",
    "service": "text_chem",
    "parameters": {
        "attribute": "drugbank_container_identifiers.drugbank_id",
        "operator": "exists",
        "negation": False
    }
}

label = "FDA-approved" if FDA_ONLY else "all DrugBank"
mode = "large & complex" if REQUIRE_COMPLEX else "large only"
print(f"Querying RCSB for {label} drugs (mode: {mode})...")

query = {
    "query": {
        "type": "group",
        "logical_operator": "and",
        "nodes": [
            fda_node,
            {
                "type": "terminal",
                "service": "text_chem",
                "parameters": {
                    "attribute": "chem_comp.formula_weight",
                    "operator": "greater",
                    "value": MIN_MW,
                    "negation": False
                }
            }
        ]
    },
    "return_type": "mol_definition",
    "request_options": {
        "paginate": {"start": 0, "rows": 5000},
        "sort": [{"sort_by": "chem_comp.formula_weight", "direction": "desc"}]
    }
}

url = "https://search.rcsb.org/rcsbsearch/v2/query"
response = requests.post(url, json=query)
results = response.json()
fda_ids = [r["identifier"] for r in results.get("result_set", [])]
print(f"Found {len(fda_ids)} {label} drugs with MW > {MIN_MW} Da")

# Step 2: Download SDF, compute size & complexity, filter
def compute_extent_from_sdf(filepath):
    """Max atom-to-atom distance (Å) from SDF coordinates."""
    coords = []
    with open(filepath) as f:
        lines = f.readlines()
    try:
        num_atoms = int(lines[3][:3])
        for i in range(4, 4 + num_atoms):
            parts = lines[i].split()
            coords.append([float(parts[0]), float(parts[1]), float(parts[2])])
    except (ValueError, IndexError):
        return 0.0
    if len(coords) < 2:
        return 0.0
    return pdist(np.array(coords)).max()

failed = []
filtered_drugs = []  # (comp_id, extent, mw, n_heavy, n_rot, n_rings, smiles)

for i, cid in enumerate(fda_ids):
    dl_url = f"https://files.rcsb.org/ligands/download/{cid}_ideal.sdf"
    r = requests.get(dl_url)
    if r.status_code != 200 or "V2000" not in r.text:
        failed.append(cid)
        continue

    filepath = f"ligands_sdf_large/{cid}_ideal.sdf"
    with open(filepath, "w") as f:
        f.write(r.text)

    # Parse with RDKit
    mol = Chem.MolFromMolFile(filepath, sanitize=True, removeHs=True)
    if mol is None:
        failed.append(cid)
        continue

    n_heavy = mol.GetNumHeavyAtoms()
    n_rot = Descriptors.NumRotatableBonds(mol)
    n_rings = rdMolDescriptors.CalcNumRings(mol)
    mw = Descriptors.MolWt(mol)
    extent = compute_extent_from_sdf(filepath)

    # Size filter (always applied)
    is_large = (n_heavy >= MIN_HEAVY_ATOMS and extent >= MIN_EXTENT)

    # Complexity filter (only when REQUIRE_COMPLEX is True)
    is_complex = (n_rot >= MIN_ROTATABLE_BONDS and n_rings >= MIN_RINGS)

    if is_large and (not REQUIRE_COMPLEX or is_complex):
        smiles = Chem.MolToSmiles(mol)
        filtered_drugs.append((cid, round(extent, 2), round(mw, 1),
                               n_heavy, n_rot, n_rings, smiles))

    if i % 50 == 0:
        print(f"Processed {i}/{len(fda_ids)} — {len(filtered_drugs)} {mode} drugs found...")
    time.sleep(0.05)

# Sort by extent descending and keep top N largest
filtered_drugs.sort(key=lambda x: -x[1])
large_drugs = filtered_drugs[:NUM_LIGANDS]

print(f"\nDone. Failed downloads: {len(failed)}")
print(f"{mode.capitalize()} {label} drugs found: {len(filtered_drugs)}")
print(f"Keeping top {NUM_LIGANDS} largest: {len(large_drugs)} drugs")
if large_drugs:
    print(f"Size range: {large_drugs[-1][1]} – {large_drugs[0][1]} Å")
print(f"\nTop 30 largest {mode} {label} drugs:")
print(f"{'ID':>8}  {'Å':>6}  {'MW':>7}  {'Atoms':>5}  {'Rot':>3}  {'Ring':>4}  SMILES")
print("-" * 80)
for cid, ext, mw, nh, nr, nring, smi in large_drugs[:30]:
    print(f"{cid:>8}  {ext:>6.1f}  {mw:>7.1f}  {nh:>5}  {nr:>3}  {nring:>4}  {smi[:40]}...")

Querying RCSB for FDA-approved drugs (mode: large only)...
Found 952 FDA-approved drugs with MW > 100 Da
Processed 0/952 — 1 large only drugs found...
Processed 50/952 — 43 large only drugs found...
Processed 100/952 — 87 large only drugs found...
Processed 150/952 — 122 large only drugs found...
Processed 200/952 — 158 large only drugs found...
Processed 250/952 — 194 large only drugs found...
Processed 300/952 — 224 large only drugs found...
Processed 350/952 — 250 large only drugs found...
Processed 400/952 — 276 large only drugs found...
Processed 450/952 — 292 large only drugs found...
Processed 500/952 — 300 large only drugs found...
Processed 550/952 — 302 large only drugs found...
Processed 600/952 — 310 large only drugs found...
Processed 650/952 — 319 large only drugs found...
Processed 700/952 — 323 large only drugs found...
Processed 750/952 — 324 large only drugs found...
Processed 800/952 — 324 large only drugs found...
Processed 850/952 — 324 large only drugs found...
Pr

In [9]:
# --- Remove SDF files for ligands that didn't pass the filters ---
import glob

kept_ids = {cid for cid, *_ in large_drugs}
sdf_files = glob.glob("ligands_sdf_large/*.sdf")
removed = 0

for filepath in sdf_files:
    filename = os.path.basename(filepath)
    cid = filename.replace("_ideal.sdf", "")
    if cid not in kept_ids:
        os.remove(filepath)
        removed += 1

remaining = len(glob.glob("ligands_sdf_large/*.sdf"))
print(f"Removed {removed} SDF files below {MIN_EXTENT} Å threshold")
print(f"Remaining in ligands_sdf_large/: {remaining} files")

Removed 624 SDF files below 15.0 Å threshold
Remaining in ligands_sdf_large/: 324 files


In [13]:
import requests
import os
import time
import numpy as np
from rdkit import Chem, RDLogger
from rdkit.Chem import Descriptors, rdMolDescriptors
from scipy.spatial.distance import pdist

# Suppress noisy RDKit warnings (2D/3D tagging, valence issues)
RDLogger.DisableLog('rdApp.*')

os.makedirs("ligands_sdf_large_complex", exist_ok=True)

# ── Configuration ──
FDA_ONLY = True             # Set to False to include all DrugBank drugs
REQUIRE_COMPLEX = True      # True = large AND complex; False = large only
NUM_LIGANDS = 335           # Number of largest drugs to keep
MIN_MW = 100                # Minimum formula weight (Da)
MIN_HEAVY_ATOMS = 15        # Minimum heavy atoms for "large"
MIN_EXTENT = 10.0           # Minimum spatial extent (Å)
MIN_ROTATABLE_BONDS = 3     # Minimum rotatable bonds for "complex" (only if REQUIRE_COMPLEX)
MIN_RINGS = 2              # Minimum ring count for "complex" (only if REQUIRE_COMPLEX)

# Step 1: Query RCSB for drugs with high MW
fda_node = {
    "type": "terminal",
    "service": "text_chem",
    "parameters": {
        "attribute": "drugbank_info.drug_groups",
        "operator": "exact_match",
        "value": "approved",
        "negation": False
    }
} if FDA_ONLY else {
    "type": "terminal",
    "service": "text_chem",
    "parameters": {
        "attribute": "drugbank_container_identifiers.drugbank_id",
        "operator": "exists",
        "negation": False
    }
}

label = "FDA-approved" if FDA_ONLY else "all DrugBank"
mode = "large & complex" if REQUIRE_COMPLEX else "large only"
print(f"Querying RCSB for {label} drugs (mode: {mode})...")

query = {
    "query": {
        "type": "group",
        "logical_operator": "and",
        "nodes": [
            fda_node,
            {
                "type": "terminal",
                "service": "text_chem",
                "parameters": {
                    "attribute": "chem_comp.formula_weight",
                    "operator": "greater",
                    "value": MIN_MW,
                    "negation": False
                }
            }
        ]
    },
    "return_type": "mol_definition",
    "request_options": {
        "paginate": {"start": 0, "rows": 5000},
        "sort": [{"sort_by": "chem_comp.formula_weight", "direction": "desc"}]
    }
}

url = "https://search.rcsb.org/rcsbsearch/v2/query"
response = requests.post(url, json=query)
results = response.json()
fda_ids = [r["identifier"] for r in results.get("result_set", [])]
print(f"Found {len(fda_ids)} {label} drugs with MW > {MIN_MW} Da")

# Step 2: Download SDF, compute size & complexity, filter
def compute_extent_from_sdf(filepath):
    """Max atom-to-atom distance (Å) from SDF coordinates."""
    coords = []
    with open(filepath) as f:
        lines = f.readlines()
    try:
        num_atoms = int(lines[3][:3])
        for i in range(4, 4 + num_atoms):
            parts = lines[i].split()
            coords.append([float(parts[0]), float(parts[1]), float(parts[2])])
    except (ValueError, IndexError):
        return 0.0
    if len(coords) < 2:
        return 0.0
    return pdist(np.array(coords)).max()

failed = []
filtered_drugs = []  # (comp_id, extent, mw, n_heavy, n_rot, n_rings, smiles)

for i, cid in enumerate(fda_ids):
    dl_url = f"https://files.rcsb.org/ligands/download/{cid}_ideal.sdf"
    r = requests.get(dl_url)
    if r.status_code != 200 or "V2000" not in r.text:
        failed.append(cid)
        continue

    filepath = f"ligands_sdf_large_complex/{cid}_ideal.sdf"
    with open(filepath, "w") as f:
        f.write(r.text)

    # Parse with RDKit
    mol = Chem.MolFromMolFile(filepath, sanitize=True, removeHs=True)
    if mol is None:
        failed.append(cid)
        continue

    n_heavy = mol.GetNumHeavyAtoms()
    n_rot = Descriptors.NumRotatableBonds(mol)
    n_rings = rdMolDescriptors.CalcNumRings(mol)
    mw = Descriptors.MolWt(mol)
    extent = compute_extent_from_sdf(filepath)

    # Size filter (always applied)
    is_large = (n_heavy >= MIN_HEAVY_ATOMS and extent >= MIN_EXTENT)

    # Complexity filter (only when REQUIRE_COMPLEX is True)
    is_complex = (n_rot >= MIN_ROTATABLE_BONDS and n_rings >= MIN_RINGS)

    if is_large and (not REQUIRE_COMPLEX or is_complex):
        smiles = Chem.MolToSmiles(mol)
        filtered_drugs.append((cid, round(extent, 2), round(mw, 1),
                               n_heavy, n_rot, n_rings, smiles))

    if i % 50 == 0:
        print(f"Processed {i}/{len(fda_ids)} — {len(filtered_drugs)} {mode} drugs found...")
    time.sleep(0.05)

# Sort by extent descending and keep top N largest
filtered_drugs.sort(key=lambda x: -x[1])
large_drugs = filtered_drugs[:NUM_LIGANDS]

print(f"\nDone. Failed downloads: {len(failed)}")
print(f"{mode.capitalize()} {label} drugs found: {len(filtered_drugs)}")
print(f"Keeping top {NUM_LIGANDS} largest: {len(large_drugs)} drugs")
if large_drugs:
    print(f"Size range: {large_drugs[-1][1]} – {large_drugs[0][1]} Å")
print(f"\nTop 30 largest {mode} {label} drugs:")
print(f"{'ID':>8}  {'Å':>6}  {'MW':>7}  {'Atoms':>5}  {'Rot':>3}  {'Ring':>4}  SMILES")
print("-" * 80)
for cid, ext, mw, nh, nr, nring, smi in large_drugs[:30]:
    print(f"{cid:>8}  {ext:>6.1f}  {mw:>7.1f}  {nh:>5}  {nr:>3}  {nring:>4}  {smi[:40]}...")

Querying RCSB for FDA-approved drugs (mode: large & complex)...
Found 952 FDA-approved drugs with MW > 100 Da
Processed 0/952 — 1 large & complex drugs found...
Processed 50/952 — 42 large & complex drugs found...
Processed 100/952 — 90 large & complex drugs found...
Processed 150/952 — 135 large & complex drugs found...
Processed 200/952 — 183 large & complex drugs found...
Processed 250/952 — 230 large & complex drugs found...
Processed 300/952 — 274 large & complex drugs found...
Processed 350/952 — 316 large & complex drugs found...
Processed 400/952 — 359 large & complex drugs found...
Processed 450/952 — 393 large & complex drugs found...
Processed 500/952 — 423 large & complex drugs found...
Processed 550/952 — 454 large & complex drugs found...
Processed 600/952 — 474 large & complex drugs found...
Processed 650/952 — 495 large & complex drugs found...
Processed 700/952 — 514 large & complex drugs found...
Processed 750/952 — 522 large & complex drugs found...
Processed 800/952

In [14]:
# --- Remove SDF files for ligands that didn't pass the filters ---
import glob

kept_ids = {cid for cid, *_ in large_drugs}
sdf_files = glob.glob("ligands_sdf_large_complex/*.sdf")
removed = 0

for filepath in sdf_files:
    filename = os.path.basename(filepath)
    cid = filename.replace("_ideal.sdf", "")
    if cid not in kept_ids:
        os.remove(filepath)
        removed += 1

remaining = len(glob.glob("ligands_sdf_large_complex/*.sdf"))
print(f"Removed {removed} SDF files below {MIN_EXTENT} Å threshold")
print(f"Remaining in ligands_sdf_large/: {remaining} files")

Removed 613 SDF files below 10.0 Å threshold
Remaining in ligands_sdf_large/: 335 files


In [19]:
import os

large_dir = "ligands_sdf_large"
complex_dir = "ligands_sdf_large_complex"

large_ids = {f.replace("_ideal.sdf", "") for f in os.listdir(large_dir) if f.endswith(".sdf")}
complex_ids = {f.replace("_ideal.sdf", "") for f in os.listdir(complex_dir) if f.endswith(".sdf")}

shared = large_ids & complex_ids
only_large = large_ids - complex_ids
only_complex = complex_ids - large_ids

print(f"ligands_sdf_large:         {len(large_ids)} ligands")
print(f"ligands_sdf_large_complex: {len(complex_ids)} ligands")
print(f"")
print(f"Exact same (in both):      {len(shared)}")
print(f"Only in large:             {len(only_large)}")
print(f"Only in large_complex:     {len(only_complex)}")
print(f"Total unique across both:  {len(large_ids | complex_ids)}")

if shared:
    print(f"\nShared ligands (first 20): {sorted(shared)[:20]}")
if only_large:
    print(f"\nOnly in large (first 20): {sorted(only_large)[:20]}")
if only_complex:
    print(f"\nOnly in complex (first 20): {sorted(only_complex)[:20]}")

ligands_sdf_large:         324 ligands
ligands_sdf_large_complex: 335 ligands

Exact same (in both):      289
Only in large:             35
Only in large_complex:     46
Total unique across both:  370

Shared ligands (first 20): ['017', '032', '04J', '07J', '08Y', '09L', '0LI', '0RI', '0WM', '0XZ', '115', '116', '117', '15M', '1C9', '1E8', '1II', '1LT', '1N1', '1UN']

Only in large (first 20): ['10A', '16A', '4DY', '87Q', 'A1JTY', 'ACD', 'AOQ', 'C41', 'CE9', 'DME', 'EIC', 'ELA', 'EPA', 'GDS', 'GSH', 'GZX', 'HXA', 'KTY', 'LNL', 'P2E']

Only in complex (first 20): ['1KX', '29S', '2HB', '3JD', '478', '4D6', '4KO', '65B', 'A1CIW', 'A1Z', 'BEP', 'BJX', 'BO2', 'BRL', 'BZ1', 'C2F', 'CHU', 'CTY', 'DM1', 'DM2']


In [22]:
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
import os
import time
import numpy as np
from rdkit import Chem, RDLogger
from rdkit.Chem import Descriptors, rdMolDescriptors

# Suppress noisy RDKit warnings
RDLogger.DisableLog('rdApp.*')

os.makedirs("ligands_sdf_small_symmetric", exist_ok=True)

# ── Configuration ──
FDA_ONLY = False             # Set to False to include all DrugBank drugs
NUM_LIGANDS = 335           # Number of small symmetric ligands to keep
MAX_MW = 300                # Maximum formula weight (Da) — defines "small"
MIN_HEAVY_ATOMS = 3         # Minimum heavy atoms (exclude trivial molecules)
MAX_HEAVY_ATOMS = 18        # Maximum heavy atoms — enforces "small"
MAX_ROTATABLE_BONDS = 2     # Maximum rotatable bonds — enforces "primitive/rigid"
MAX_RINGS = 2               # Maximum ring count — enforces "primitive"
MAX_SYMMETRY_RATIO = 0.5    # Lower = more symmetric (unique environments / total heavy atoms)
                            # Benzene = 0.17, Aspirin ≈ 1.0, Naphthalene = 0.2
MAX_CHIRAL_CENTERS = 0      # Max stereocenters — 0 = fully achiral (more symmetric)

# Step 1: Query RCSB for small drugs
fda_node = {
    "type": "terminal",
    "service": "text_chem",
    "parameters": {
        "attribute": "drugbank_info.drug_groups",
        "operator": "exact_match",
        "value": "approved",
        "negation": False
    }
} if FDA_ONLY else {
    "type": "terminal",
    "service": "text_chem",
    "parameters": {
        "attribute": "drugbank_container_identifiers.drugbank_id",
        "operator": "exists",
        "negation": False
    }
}

label = "FDA-approved" if FDA_ONLY else "all DrugBank"
print(f"Querying RCSB for small, primitive & symmetric {label} drugs...")
print(f"  MW ≤ {MAX_MW} Da | Atoms {MIN_HEAVY_ATOMS}–{MAX_HEAVY_ATOMS} | "
      f"Rot ≤ {MAX_ROTATABLE_BONDS} | Rings ≤ {MAX_RINGS} | "
      f"Sym ratio ≤ {MAX_SYMMETRY_RATIO} | Chiral ≤ {MAX_CHIRAL_CENTERS}")

query = {
    "query": {
        "type": "group",
        "logical_operator": "and",
        "nodes": [
            fda_node,
            {
                "type": "terminal",
                "service": "text_chem",
                "parameters": {
                    "attribute": "chem_comp.formula_weight",
                    "operator": "less",
                    "value": MAX_MW,
                    "negation": False
                }
            }
        ]
    },
    "return_type": "mol_definition",
    "request_options": {
        "paginate": {"start": 0, "rows": 5000},
        "sort": [{"sort_by": "chem_comp.formula_weight", "direction": "asc"}]
    }
}

url = "https://search.rcsb.org/rcsbsearch/v2/query"
response = requests.post(url, json=query)
results = response.json()
candidate_ids = [r["identifier"] for r in results.get("result_set", [])]
print(f"Found {len(candidate_ids)} {label} drugs with MW < {MAX_MW} Da")

# Step 2: Download SDF, parse with RDKit, filter for small + primitive + symmetric
session = requests.Session()
retries = Retry(total=5, backoff_factor=1.0,
                status_forcelist=[429, 500, 502, 503, 504],
                allowed_methods=["GET"])
session.mount("https://", HTTPAdapter(max_retries=retries, pool_maxsize=10))

def compute_symmetry_ratio(mol):
    """Ratio of unique atomic environments to total heavy atoms.
    Lower = more symmetric. Benzene=0.17, fully asymmetric=1.0."""
    n_heavy = mol.GetNumHeavyAtoms()
    if n_heavy == 0:
        return 1.0
    ranks = Chem.CanonicalRankAtoms(mol, breakTies=False)
    n_unique = len(set(ranks))
    return n_unique / n_heavy

failed = []
filtered_ligands = []  # (comp_id, sym_ratio, mw, n_heavy, n_rot, n_rings, n_chiral, smiles)

for i, cid in enumerate(candidate_ids):
    dl_url = f"https://files.rcsb.org/ligands/download/{cid}_ideal.sdf"
    try:
        r = session.get(dl_url, timeout=15)
    except requests.exceptions.RequestException:
        failed.append(cid)
        continue

    if r.status_code != 200 or "V2000" not in r.text:
        failed.append(cid)
        continue

    # Save SDF
    filepath = f"ligands_sdf_small_symmetric/{cid}_ideal.sdf"
    with open(filepath, "w") as f:
        f.write(r.text)

    # Parse with RDKit
    mol = Chem.MolFromMolFile(filepath, sanitize=True, removeHs=True)
    if mol is None:
        failed.append(cid)
        continue

    n_heavy = mol.GetNumHeavyAtoms()
    mw = Descriptors.MolWt(mol)
    n_rot = Descriptors.NumRotatableBonds(mol)
    n_rings = rdMolDescriptors.CalcNumRings(mol)
    n_chiral = len(Chem.FindMolChiralCenters(mol, includeUnassigned=True))

    # Filter: small
    if n_heavy < MIN_HEAVY_ATOMS or n_heavy > MAX_HEAVY_ATOMS:
        continue

    # Filter: primitive (rigid, few rings)
    if n_rot > MAX_ROTATABLE_BONDS:
        continue
    if n_rings > MAX_RINGS:
        continue

    # Filter: symmetric (low unique-environment ratio, achiral)
    sym_ratio = compute_symmetry_ratio(mol)
    if sym_ratio > MAX_SYMMETRY_RATIO:
        continue
    if n_chiral > MAX_CHIRAL_CENTERS:
        continue

    smiles = Chem.MolToSmiles(mol)
    filtered_ligands.append((cid, round(sym_ratio, 3), round(mw, 1),
                             n_heavy, n_rot, n_rings, n_chiral, smiles))

    if i % 100 == 0:
        print(f"Processed {i}/{len(candidate_ids)} — "
              f"{len(filtered_ligands)} small symmetric primitives found...")
    time.sleep(0.15)

# Sort by symmetry ratio (most symmetric first), then by fewest atoms
filtered_ligands.sort(key=lambda x: (x[1], x[3]))
kept_ligands = filtered_ligands[:NUM_LIGANDS]

print(f"\nDone. Failed downloads: {len(failed)}")
print(f"Small, primitive & symmetric {label} drugs found: {len(filtered_ligands)}")
print(f"Keeping top {NUM_LIGANDS}: {len(kept_ligands)} drugs")
if kept_ligands:
    print(f"Symmetry ratio range: {kept_ligands[0][1]} – {kept_ligands[-1][1]}")

print(f"\nTop 30 most symmetric small primitives:")
print(f"{'ID':>8}  {'Sym':>5}  {'MW':>7}  {'Atoms':>5}  {'Rot':>3}  {'Ring':>4}  {'Chir':>4}  SMILES")
print("-" * 85)
for cid, sym, mw, nh, nr, nring, nchi, smi in kept_ligands[:30]:
    print(f"{cid:>8}  {sym:>5.3f}  {mw:>7.1f}  {nh:>5}  {nr:>3}  {nring:>4}  {nchi:>4}  {smi}")

Querying RCSB for small, primitive & symmetric all DrugBank drugs...
  MW ≤ 300 Da | Atoms 3–18 | Rot ≤ 2 | Rings ≤ 2 | Sym ratio ≤ 0.5 | Chiral ≤ 0
Found 2739 all DrugBank drugs with MW < 300 Da

Done. Failed downloads: 1
Small, primitive & symmetric all DrugBank drugs found: 48
Keeping top 335: 48 drugs
Symmetry ratio range: 0.125 – 0.5

Top 30 most symmetric small primitives:
      ID    Sym       MW  Atoms  Rot  Ring  Chir  SMILES
-------------------------------------------------------------------------------------
     PS9  0.125    256.5      8    0     1     0  S1SSSSSSS1
     2H3  0.167    180.2     12    0     1     0  O[C@H]1[C@H](O)[C@@H](O)[C@H](O)[C@@H](O)[C@@H]1O
     NCO  0.286    161.1      7    0     0     0  [NH3]->[Co@OH25+3](<-[NH3])(<-[NH3])(<-[NH3])(<-[NH3])<-[NH3]
     PZE  0.333     86.1      6    0     1     0  C1CNCCN1
     DIO  0.333     88.1      6    0     1     0  C1COCCO1
     13X  0.333    126.1      9    0     1     0  Oc1cc(O)cc(O)c1
     TCZ  0.333   

In [23]:
# --- Remove SDF files for ligands that didn't pass the filters ---
import glob

kept_ids = {cid for cid, *_ in kept_ligands}
sdf_files = glob.glob("ligands_sdf_small_symmetric/*.sdf")
removed = 0

for filepath in sdf_files:
    filename = os.path.basename(filepath)
    cid = filename.replace("_ideal.sdf", "")
    if cid not in kept_ids:
        os.remove(filepath)
        removed += 1

remaining = len(glob.glob("ligands_sdf_small_symmetric/*.sdf"))
print(f"Removed {removed} SDF files that didn't pass filters")
print(f"Remaining in ligands_sdf_small_symmetric/: {remaining} files")

Removed 2691 SDF files that didn't pass filters
Remaining in ligands_sdf_small_symmetric/: 48 files


In [ ]:
import requests
import os
import time
import numpy as np
from scipy.spatial.distance import pdist

os.makedirs("ligands_sdf", exist_ok=True)

# ── Configuration ──
SIZE_THRESHOLD = 20.0   # Minimum extent in Angstroms
NUM_LIGANDS = 335       # Number of top ligands to keep (sorted by size, descending)

def download_ligand_sdf(comp_id):
    """Download ideal coordinates SDF from RCSB CCD."""
    url = f"https://files.rcsb.org/ligands/download/{comp_id}_ideal.sdf"
    r = requests.get(url)
    if r.status_code == 200 and "V2000" in r.text:
        filepath = f"ligands_sdf/{comp_id}_ideal.sdf"
        with open(filepath, "w") as f:
            f.write(r.text)
        return filepath
    return None

def compute_max_extent_sdf(sdf_path):
    """Compute the maximum atom-to-atom distance (Å) from an SDF file."""
    coords = []
    with open(sdf_path) as f:
        lines = f.readlines()
    try:
        num_atoms = int(lines[3][:3])
        for i in range(4, 4 + num_atoms):
            parts = lines[i].split()
            x, y, z = float(parts[0]), float(parts[1]), float(parts[2])
            coords.append([x, y, z])
    except (ValueError, IndexError):
        return 0.0
    if len(coords) < 2:
        return 0.0
    return pdist(np.array(coords)).max()

# Download and measure all FDA-approved drug candidates
failed = []
all_measured = []  # (comp_id, extent_angstrom)

for i, cid in enumerate(comp_ids):
    sdf_path = download_ligand_sdf(cid)
    if sdf_path is None:
        failed.append(cid)
    else:
        extent = compute_max_extent_sdf(sdf_path)
        if extent > SIZE_THRESHOLD:
            all_measured.append((cid, round(extent, 2)))
    if i % 100 == 0:
        print(f"Processed {i}/{len(comp_ids)} — {len(all_measured)} drugs > {SIZE_THRESHOLD} Å so far...")
    time.sleep(0.05)

# Sort by extent descending and keep top N
all_measured.sort(key=lambda x: -x[1])
large_ligands = all_measured[:NUM_LIGANDS]

print(f"\nDone. Failed downloads: {len(failed)}")
print(f"FDA-approved drugs with extent > {SIZE_THRESHOLD} Å: {len(all_measured)}")
print(f"Keeping top {NUM_LIGANDS}: {len(large_ligands)} drugs")
print(f"\nSize range: {large_ligands[-1][1]} – {large_ligands[0][1]} Å" if large_ligands else "")
print("\nFirst 20 largest FDA-approved drugs (ID, max extent in Å):")
for cid, ext in large_ligands[:20]:
    print(f"  {cid}: {ext} Å")

Processed 0/1500 — 0 drugs > 20.0 Å so far...
Processed 100/1500 — 74 drugs > 20.0 Å so far...
Processed 200/1500 — 139 drugs > 20.0 Å so far...
Processed 300/1500 — 191 drugs > 20.0 Å so far...
Processed 400/1500 — 226 drugs > 20.0 Å so far...
Processed 500/1500 — 253 drugs > 20.0 Å so far...
Processed 600/1500 — 282 drugs > 20.0 Å so far...
Processed 700/1500 — 304 drugs > 20.0 Å so far...
Processed 800/1500 — 319 drugs > 20.0 Å so far...
Processed 900/1500 — 337 drugs > 20.0 Å so far...
Processed 1000/1500 — 351 drugs > 20.0 Å so far...
Processed 1100/1500 — 362 drugs > 20.0 Å so far...
Processed 1200/1500 — 374 drugs > 20.0 Å so far...
Processed 1300/1500 — 386 drugs > 20.0 Å so far...
Processed 1400/1500 — 398 drugs > 20.0 Å so far...

Done. Failed downloads: 28
FDA-approved drugs with extent > 20.0 Å: 407
Keeping top 335: 335 drugs

Size range: 20.78 – 61.3 Å

First 20 largest FDA-approved drugs (ID, max extent in Å):
  CDL: 61.3 Å
  DR6: 53.9 Å
  BV4: 50.47 Å
  AGH: 46.84 Å
  SF

In [ ]:
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
import os
import time
import numpy as np
from rdkit import Chem, RDLogger

# Suppress noisy RDKit warnings
RDLogger.DisableLog('rdApp.*')

os.makedirs("ligands_sdf_small", exist_ok=True)

# ── Configuration ──
FDA_ONLY = False             # Set to False to include all DrugBank drugs
NUM_SMALL_LIGANDS = 335     # Number of small symmetric ligands to keep
MAX_MW = 300                # Maximum formula weight (Da) — defines "small"
MAX_HEAVY_ATOMS = 20        # Maximum number of heavy atoms
MAX_SYMMETRY_RATIO = 0.75   # Lower = more symmetric (unique environments / total heavy atoms)
                            # Benzene = 0.17, Aspirin = 1.0, Naphthalene = 0.2
                            # 0.75 is relaxed — top N are kept sorted by ratio anyway

# Step 1: Query RCSB for small drugs
fda_node = {
    "type": "terminal",
    "service": "text_chem",
    "parameters": {
        "attribute": "drugbank_info.drug_groups",
        "operator": "exact_match",
        "value": "approved",
        "negation": False
    }
} if FDA_ONLY else {
    "type": "terminal",
    "service": "text_chem",
    "parameters": {
        "attribute": "drugbank_container_identifiers.drugbank_id",
        "operator": "exists",
        "negation": False
    }
}

label = "FDA-approved" if FDA_ONLY else "all DrugBank"
print(f"Querying RCSB for small & symmetric {label} drugs...")

query = {
    "query": {
        "type": "group",
        "logical_operator": "and",
        "nodes": [
            fda_node,
            {
                "type": "terminal",
                "service": "text_chem",
                "parameters": {
                    "attribute": "chem_comp.formula_weight",
                    "operator": "less",
                    "value": MAX_MW,
                    "negation": False
                }
            }
        ]
    },
    "return_type": "mol_definition",
    "request_options": {
        "paginate": {"start": 0, "rows": 5000},
        "sort": [{"sort_by": "chem_comp.formula_weight", "direction": "asc"}]
    }
}

url = "https://search.rcsb.org/rcsbsearch/v2/query"
response = requests.post(url, json=query)
results = response.json()
small_ids = [r["identifier"] for r in results.get("result_set", [])]
print(f"Found {len(small_ids)} small {label} drugs (MW < {MAX_MW} Da)")

# Step 2: Download SDF, parse with RDKit, and filter for symmetry
# Set up a session with automatic retries for transient network errors
session = requests.Session()
retries = Retry(total=5, backoff_factor=1.0,
                status_forcelist=[429, 500, 502, 503, 504],
                allowed_methods=["GET"])
session.mount("https://", HTTPAdapter(max_retries=retries, pool_maxsize=10))

def compute_symmetry_ratio(mol):
    """Ratio of unique atomic environments to total heavy atoms.
    Lower values = more symmetric. Benzene=0.17, fully asymmetric=1.0."""
    n_heavy = mol.GetNumHeavyAtoms()
    if n_heavy == 0:
        return 1.0
    ranks = Chem.CanonicalRankAtoms(mol, breakTies=False)
    n_unique = len(set(ranks))
    return n_unique / n_heavy

failed = []
symmetric_ligands = []  # (comp_id, symmetry_ratio, n_heavy_atoms, smiles)

for i, cid in enumerate(small_ids):
    dl_url = f"https://files.rcsb.org/ligands/download/{cid}_ideal.sdf"
    try:
        r = session.get(dl_url, timeout=15)
    except requests.exceptions.RequestException:
        failed.append(cid)
        continue

    if r.status_code != 200 or "V2000" not in r.text:
        failed.append(cid)
        continue

    # Save SDF to ligands_sdf_small/
    filepath = f"ligands_sdf_small/{cid}_ideal.sdf"
    with open(filepath, "w") as f:
        f.write(r.text)

    # Parse with RDKit
    mol = Chem.MolFromMolFile(filepath, sanitize=True, removeHs=True)
    if mol is None:
        failed.append(cid)
        continue

    n_heavy = mol.GetNumHeavyAtoms()
    if n_heavy < 3 or n_heavy > MAX_HEAVY_ATOMS:
        continue

    sym_ratio = compute_symmetry_ratio(mol)
    if sym_ratio <= MAX_SYMMETRY_RATIO:
        smiles = Chem.MolToSmiles(mol)
        symmetric_ligands.append((cid, round(sym_ratio, 3), n_heavy, smiles))

    if i % 100 == 0:
        print(f"Processed {i}/{len(small_ids)} — {len(symmetric_ligands)} symmetric {label} drugs found...")
    time.sleep(0.15)  # Gentler rate to avoid server resets

# Sort by symmetry ratio (most symmetric first) and keep top N
symmetric_ligands.sort(key=lambda x: x[1])
small_sym_ligands = symmetric_ligands[:NUM_SMALL_LIGANDS]

print(f"\nDone. Failed: {len(failed)}")
print(f"Symmetric {label} drugs found (ratio ≤ {MAX_SYMMETRY_RATIO}): {len(symmetric_ligands)}")
print(f"Keeping top {NUM_SMALL_LIGANDS}: {len(small_sym_ligands)} drugs")
print(f"\nFirst 30 most symmetric small {label} drugs:")
print(f"{'ID':>8}  {'Sym':>5}  {'Atoms':>5}  SMILES")
print("-" * 60)
for cid, sym, n, smi in small_sym_ligands[:30]:
    print(f"{cid:>8}  {sym:>5.3f}  {n:>5}  {smi}")

Querying RCSB for small all DrugBank drugs...
Found 2739 small all DrugBank drugs (MW < 300 Da)
Processed 50/2739 — 3 symmetric all DrugBank drugs found...
Processed 100/2739 — 6 symmetric all DrugBank drugs found...
Processed 150/2739 — 13 symmetric all DrugBank drugs found...
Processed 200/2739 — 14 symmetric all DrugBank drugs found...
Processed 250/2739 — 19 symmetric all DrugBank drugs found...
Processed 300/2739 — 26 symmetric all DrugBank drugs found...
Processed 350/2739 — 28 symmetric all DrugBank drugs found...
Processed 400/2739 — 29 symmetric all DrugBank drugs found...
Processed 450/2739 — 32 symmetric all DrugBank drugs found...
Processed 500/2739 — 34 symmetric all DrugBank drugs found...


ConnectionError: ('Connection aborted.', ConnectionResetError(104, 'Connection reset by peer'))

In [ ]:
# --- Remove SDF files for ligands that didn't pass the filters ---
import glob

kept_ids = {cid for cid, *_ in small_sym_ligands}
sdf_files = glob.glob("ligands_sdf_small/*.sdf")
removed = 0

for filepath in sdf_files:
    filename = os.path.basename(filepath)
    cid = filename.replace("_ideal.sdf", "")
    if cid not in kept_ids:
        os.remove(filepath)
        removed += 1

remaining = len(glob.glob("ligands_sdf_small/*.sdf"))
print(f"Removed {removed} SDF files that didn't pass symmetry filter")
print(f"Remaining in ligands_sdf_small/: {remaining} files")